# Data flow — one paper, every stageFollows a single paper from the PDF all the way to a retrieved chunk, printing what thedata actually looks like at each step. Nothing here changes state — it reads what'salready in Postgres and OpenSearch.```PDF ──Docling──> sections + raw_text ──Postgres──> chunks ──embed──> OpenSearch ──search──> agent        (1)              (2)                        (3)      (4)        (5)         (6)```**Prereqs:** `docker compose up -d postgres opensearch`, a populated index, `OPENAI_API_KEY`.Cheap to run — one embedding call total (~$0.000002).

In [ ]:
from dotenv import load_dotenvload_dotenv()# Import the ORM model before anything builds the DB (CLAUDE.md gotcha #1)from src.models.paper import Paper  # noqa: F401from src.db.factory import make_databasefrom src.repositories.paper import PaperRepositorydb = make_database()with db.get_session() as s:    repo = PaperRepository(s)    total = repo.get_count()    parsed = repo.get_papers_with_raw_text(limit=500)print(f"papers in Postgres : {total}")print(f"with parsed text   : {len(parsed)}  <- only these can be chunked")print(f"unparseable        : {total - len(parsed)}  (PDF over 30 pages or 20MB)")print()print("pick any of these:")for p in parsed[:5]:    print(f"  {p.arxiv_id}  {p.title[:60]}")ARXIV_ID = parsed[0].arxiv_id   # <- change this to follow a different paperprint(f"\nfollowing: {ARXIV_ID}")

---## Stages 1-2 — Docling parsed the PDF into `raw_text` + `sections`This already happened during ingestion. Docling read the PDF and produced two thingsstored on the `papers` row:- **`raw_text`** — the whole paper as one flat string- **`sections`** — a list of `{title, content}`, i.e. Docling's reading of the paper's  own headings`sections` is the interesting one: it's what lets the chunker follow the document'sstructure instead of blindly slicing every 600 words.

In [ ]:
with db.get_session() as s:    paper = PaperRepository(s).get_by_arxiv_id(ARXIV_ID)    title, abstract = paper.title, paper.abstract    raw_text, sections = paper.raw_text, paper.sections    paper_db_id = str(paper.id)print("TITLE     :", title)print("raw_text  :", f"{len(raw_text.split()):,} words")print("sections  :", type(sections).__name__, f"of {len(sections)} items")print("item shape:", list(sections[0].keys()))print()print("section titles Docling found:")for sec in sections[:12]:    print(f"  {len(str(sec['content']).split()):>5}w  {sec['title'][:60]}")if len(sections) > 12:    print(f"  ... and {len(sections) - 12} more")

---## Stage 3 — Chunking`TextChunker.chunk_paper()` picks one of two strategies:- **section-aware** (preferred) — uses the `sections` above- **sliding window** (fallback) — only if `sections` is missing or unusableThe section-aware path buckets each section by word count:| section size | what happens | how you spot it ||---|---|---|| < 100 words | merged with neighbours | title ends `+ Combined` || 100-800 words | becomes one chunk | plain section title || > 800 words | split by sliding window | title ends `(Part N)` |The abstract always gets its own chunk (index 0). It is deliberately **not** prepended toevery chunk — that would make abstract terms match every chunk of the paper and wreckBM25 scoring.

In [ ]:
from src.services.indexing.text_chunker import TextChunkerchunker = TextChunker()   # 600 words, 100 overlap, min 100chunks = chunker.chunk_paper(    title=title, abstract=abstract, full_text=raw_text,    arxiv_id=ARXIV_ID, paper_id=paper_db_id, sections=sections,)print(f"{len(raw_text.split()):,} words  ->  {len(chunks)} chunks\n")print(f"{'idx':>3} {'words':>6}  {'rule':<10} section")print("-" * 78)for c in chunks:    st = c.metadata.section_title or "(no section)"    rule = "combined" if "+ Combined" in st else "split" if "(Part" in st else "abstract" if st == "Abstract" else "as-is"    print(f"{c.metadata.chunk_index:>3} {c.metadata.word_count:>6}  {rule:<10} {st[:52]}")

In [ ]:
# What a chunk actually contains. Note the paper title is prefixed to every chunk, so a# chunk retrieved on its own still says which paper it came from.c = chunks[1]print(f"chunk {c.metadata.chunk_index} | section={c.metadata.section_title!r} | {c.metadata.word_count} words")print("-" * 78)print(c.text[:700], "...")

---## Stage 4 — EmbeddingEach chunk's text becomes a vector. `text-embedding-3-small` at 1024 dimensions.This is the only step here that costs money and hits the network.

In [ ]:
from src.services.embeddings.factory import make_openai_embeddings_clientemb_client = make_openai_embeddings_client()vec = await emb_client.embed_text(chunks[1].text)print("model     :", emb_client.settings.model)print("dimensions:", len(vec))print("first 8   :", [round(x, 4) for x in vec[:8]])print()print("That vector is what vector search compares against. The chunk TEXT is kept too —")print("BM25 searches the text, kNN searches the vector, and RRF fuses the two rankings.")

---## Stage 5 — What's actually stored in OpenSearchOne document **per chunk**, not per paper. Each carries the chunk text, its vector, and acopy of the paper's metadata.That metadata duplication is deliberate: it means a search hit is self-contained, so acitation can be rendered straight from the hit with no Postgres lookup.

In [ ]:
from src.services.opensearch.factory import make_opensearch_clientos_client = make_opensearch_client()stored = os_client.get_chunks_by_paper(ARXIV_ID)print(f"{len(stored)} chunks indexed for {ARXIV_ID}\n")doc = stored[0]for k, v in doc.items():    if isinstance(v, list) and v and isinstance(v[0], float):        print(f"  {k:18} vector[{len(v)}]")    else:        print(f"  {k:18} {str(v)[:62]}")

---## Stage 6 — Search, three waysThe point of hybrid search: BM25 and vector search fail in **different directions**.- BM25 is exact-token matching — nails IDs and specific method names, misses paraphrase- Vector search is semantic — handles paraphrase, can miss an exact stringTheir scores are on incompatible scales (BM25 is unbounded, cosine is 0-1), so you can'taverage them. **RRF** throws the scores away and fuses on rank position instead:```score = 1 / (60 + rank)      summed across both lists```

In [ ]:
QUERY = "safety mechanism for humanoid robots"   # <- try changing thisq_vec = await emb_client.embed_text(QUERY)bm25   = os_client.search_papers(query=QUERY, size=3, latest=False)hybrid = os_client.search_unified(query=QUERY, query_embedding=q_vec, size=3, use_hybrid=True)def show(label, res):    print(f"--- {label} ---")    for h in res["hits"]:        print(f"  {h['score']:8.4f}  {h['arxiv_id']:16} {h.get('title','')[:46]}")    print()show("BM25 only (raw Lucene scores, unbounded)", bm25)show("Hybrid + RRF (rank-fused)", hybrid)print("Notice the hybrid scores are tiny and evenly spaced — they are NOT similarities.")for rank in (1, 2, 3):    print(f"   rank {rank} -> 1/(60+{rank}) = {1/(60+rank):.7f}")

---## Stage 7 — What the agent actually callsThe agent never touches OpenSearch directly. It calls the `retrieve_papers` tool, whichwraps everything above and returns **two** things:- `content` — flat text the LLM reads- `artifact` — the real `Document` objects your code readsThat split matters. Before it existed, the tool returned raw `Document`s, LangChainstringified them into the message, and the structured metadata became unreachable — theAPI returned empty `sources` on every request while the answer still cited papers inline.

In [ ]:
from src.services.agents.tools import create_retriever_tooltool = create_retriever_tool(    opensearch_client=os_client, embeddings_client=emb_client,    top_k=3, use_hybrid=True,)msg = await tool.ainvoke({"args": {"query": QUERY}, "id": "demo",                          "name": "retrieve_papers", "type": "tool_call"})print("=== .content  (what the LLM sees) ===")print(msg.content[:600], "...\n")print("=== .artifact (what your code sees) ===")for d in msg.artifact:    m = d.metadata    print(f"  {m['arxiv_id']:16} score={m['score']:.4f}  {m['title'][:44]}")print()print("grade_document_node turns .artifact into SourceItem objects ->")print("that is where the citation cards in the UI come from.")

---## Recap```PDF ──Docling──> sections + raw_text          stored in Postgres                      │                      ├─ chunk_paper()         section-aware, ~15-30 chunks                      │      │                      │      └─ embed_text()   1024-dim vector per chunk                      │             │                      │             └─ bulk_index_chunks()   one OpenSearch doc per chunk                      │search ──BM25 + kNN──> RRF fuse ──top_k=3──> retrieve_papers tool                                                  │                                    content (LLM) + artifact (code)```**Two failure modes worth remembering, both of which you can see in the numbers above:**1. If Docling rejects the PDF (>30 pages or >20MB), `raw_text` and `sections` are null →   zero chunks → the paper exists in Postgres but is **invisible to search**. 8 of the   29 papers are in that state.2. Postgres and OpenSearch can drift, because indexing is a separate step from storing.   A paper can be stored with no chunks indexed and nothing detects it.